In [4]:
from google.colab import files

uploaded = files.upload()

Saving clean_population_data.csv to clean_population_data.csv


In [7]:
import pandas as pd
import sqlite3

# Load the cleaned CSV
df = pd.read_csv("clean_population_data.csv")

# Connect to SQLite (no installation required)
conn = sqlite3.connect("population.db")

# Push data to SQLite
df.to_sql("population_data", conn, index=False, if_exists="replace")

217

In [13]:
# Top 10 countries with the highest risk
High_Risk_Countries = """
SELECT country, population_risk_score
FROM population_data
ORDER BY population_risk_score DESC
LIMIT 10;
"""
top_risk = pd.read_sql_query(High_Risk_Countries, conn)

print(top_risk)

                country  population_risk_score
0                Monaco               2.258440
1              Pakistan               2.043559
2                 Japan               2.028897
3               Germany               2.027762
4                Poland               1.984333
5      Puerto Rico (US)               1.978276
6               Finland               1.967339
7  Hong Kong SAR, China               1.940162
8                Latvia               1.930767
9                Greece               1.920142


In [15]:
Low_Fertility_Rate = """SELECT country, fertility_rate
FROM population_data
ORDER BY fertility_rate ASC
LIMIT 10;
"""
Low_fertility = pd.read_sql_query(Low_Fertility_Rate, conn)

print(Low_fertility)

                  country  fertility_rate
0        Macao SAR, China           0.686
1    Hong Kong SAR, China           0.735
2             Korea, Rep.           0.749
3        Puerto Rico (US)           0.937
4               Singapore           0.963
5                 Ukraine           0.997
6                   China           1.021
7  British Virgin Islands           1.057
8                 Curacao           1.070
9                 Andorra           1.096


In [46]:
High_Dependency_Ratio = """SELECT country, dependency_ratio
FROM population_data
ORDER BY dependency_ratio DESC
LIMIT 10
"""

High_dependency = pd.read_sql_query(High_Dependency_Ratio, conn)

print(High_dependency)

                    country  dependency_ratio
0  Central African Republic        103.585627
1                    Monaco         99.132648
2        Somalia, Fed. Rep.         96.739437
3          Congo, Dem. Rep.         96.207983
4                     Niger         95.481882
5                      Mali         92.890010
6                      Chad         92.606111
7                    Angola         88.655783
8                Mozambique         88.332455
9                   Burundi         87.018538


In [17]:
Low_migration = """SELECT country, net_migration
FROM population_data
WHERE net_migration < 0
ORDER BY net_migration ASC
LIMIT 10"""

low_migration = pd.read_sql_query(Low_migration, conn)

print(low_migration)

              country  net_migration
0            Pakistan       -1235336
1               India        -495753
2          Bangladesh        -402100
3               Nepal        -364699
4             Germany        -334072
5              Poland        -330820
6              Jordan        -303267
7               China        -268126
8             Turkiye        -258205
9  Russian Federation        -251822


In [22]:
average_population_risk = """
SELECT country, AVG(population_risk_score) AS average_risk
FROM population_data

"""
average_population_risk = pd.read_sql_query(average_population_risk, conn)

print(average_population_risk)

       country  average_risk
0  Afghanistan      1.712265


In [47]:
risk_score_categorization = """SELECT country,
       dependency_ratio,
       fertility_rate,
       net_migration,
       population_risk_score,
       CASE
           WHEN population_risk_score >= 2 THEN 'High Risk'
           WHEN population_risk_score >= 1.5 THEN 'Medium Risk'
           ELSE 'Low Risk'
       END AS risk_level
FROM population_data
ORDER BY population_risk_score DESC
LIMIT 10;"""

risk_score_categorization = pd.read_sql_query(risk_score_categorization, conn)

print(risk_score_categorization)


                country  dependency_ratio  fertility_rate  net_migration  \
0                Monaco         99.132648           2.092            100   
1              Pakistan         68.367073           3.495       -1235336   
2                 Japan         70.162064           1.225         140579   
3               Germany         60.228420           1.455        -334072   
4                Poland         54.354838           1.308        -330820   
5      Puerto Rico (US)         57.511376           0.937           5495   
6               Finland         62.686304           1.297          18246   
7  Hong Kong SAR, China         51.466228           0.735          17863   
8                Latvia         59.699224           1.347          -7330   
9                Greece         59.365894           1.339          16636   

   population_risk_score   risk_level  
0               2.258440    High Risk  
1               2.043559    High Risk  
2               2.028897    High Risk  
3  

In [44]:
fertility_replacement = """SELECT country,
       fertility_rate,
       CASE WHEN fertility_rate < 2.1 THEN 'Below Replacement' ELSE 'Above Replacement' END AS fertility_status
FROM population_data
ORDER BY fertility_rate ASC
LIMIT 10;"""

fertility_replacement = pd.read_sql_query(fertility_replacement, conn)

print(fertility_replacement)

                  country  fertility_rate   fertility_status
0        Macao SAR, China           0.686  Below Replacement
1    Hong Kong SAR, China           0.735  Below Replacement
2             Korea, Rep.           0.749  Below Replacement
3        Puerto Rico (US)           0.937  Below Replacement
4               Singapore           0.963  Below Replacement
5                 Ukraine           0.997  Below Replacement
6                   China           1.021  Below Replacement
7  British Virgin Islands           1.057  Below Replacement
8                 Curacao           1.070  Below Replacement
9                 Andorra           1.096  Below Replacement


In [45]:
largest_risk = """SELECT country,
       CASE
           WHEN dependency_ratio >= 60 THEN 'Dependency is main risk'
           WHEN fertility_rate < 2.1 THEN 'Low fertility is main risk'
           WHEN net_migration < 0 THEN 'Migration is main risk'
           ELSE 'No single factor dominates'
       END AS main_risk_factor
FROM population_data
ORDER BY population_risk_score DESC
LIMIT 10;"""

largest_risk = pd.read_sql_query(largest_risk, conn)

print(largest_risk)

                country            main_risk_factor
0                Monaco     Dependency is main risk
1              Pakistan     Dependency is main risk
2                 Japan     Dependency is main risk
3               Germany     Dependency is main risk
4                Poland  Low fertility is main risk
5      Puerto Rico (US)  Low fertility is main risk
6               Finland     Dependency is main risk
7  Hong Kong SAR, China  Low fertility is main risk
8                Latvia  Low fertility is main risk
9                Greece  Low fertility is main risk
